# 15.05 · Long-running & coding agents

**Chapter:** Advanced & Frontier Topics · **Milestone:** M7

## Goals
- [ ] Know the anatomy of an **agent harness**: loop, tools, permissions, context management, state on disk
- [ ] Make an agent survive beyond one context window: **progress file + feature list + resumable sessions**
- [ ] Implement **context compaction**, **sub-agents** (fresh context for a sub-task) and **skills** (instructions loaded on demand)
- [ ] Understand how coding agents (Claude Code, Codex, Cursor, OpenHands, Aider) are built, and how they're evaluated (SWE-bench)

## Resources
- [Anthropic – Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents) — free
- [Anthropic – Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents) ·
  [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents) — free
- [Agent Skills](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills) · [Claude Agent SDK](https://docs.claude.com/en/api/agent-sdk/overview) — free
- [SWE-agent paper (agent-computer interfaces)](https://arxiv.org/abs/2405.15793) · [OpenHands](https://github.com/All-Hands-AI/OpenHands) · [SWE-bench](https://www.swebench.com/) — free

09.02 built an agent loop that lives inside one conversation. Real tasks (a feature, a migration, a research report) take hours and
hundreds of tool calls: far more than one context window. Long-running agents treat the context window as **working memory** and the
**file system as long-term memory**:

| Problem | Harness technique |
|---|---|
| Context fills up | **compaction**: summarise old turns; keep files, not transcripts |
| Work spans sessions | **progress file** + **feature list** + tests: a new session reads them and continues |
| Sub-tasks pollute context | **sub-agents**: fresh context, return only a short report |
| Too many instructions | **skills**: list names + one-line descriptions; load the full text only when needed |
| Agent claims "done" too early | **verifiable goals**: tests that must pass, checked by the harness |
| Agent breaks things | **permissions** + sandbox (09.08), workspace-only file access, timeouts |

In [ ]:
import inspect
import json
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

from aieng import DATA
from aieng.llm import available, client

PROVIDER = "gemini" if "gemini" in available() else "groq" if "groq" in available() else "ollama"
c, MODEL = client(PROVIDER)
EXTRA = {"reasoning_effort": "none"} if PROVIDER == "ollama" else {}
print("agent model:", PROVIDER, MODEL)

## 1. The project: a small library with a test suite and a feature list
The harness gives the agent a **verifiable** goal (tests) and a machine-readable **feature list** it updates — the pattern from Anthropic's
long-running-agents post: the agent may only flip a feature to `passes: true`, never edit the tests.

In [ ]:
WS = DATA / "long-agent"
shutil.rmtree(WS, ignore_errors=True); WS.mkdir()

(WS / "textstats.py").write_text('''"""Small text utilities. Implement every function so the tests in test_textstats.py pass."""


def word_count(text: str) -> int:
    raise NotImplementedError


def top_words(text: str, n: int = 3) -> list[tuple[str, int]]:
    """Most frequent lowercase words (letters/digits/apostrophes only), ties broken alphabetically."""
    raise NotImplementedError


def sentence_count(text: str) -> int:
    """Sentences end with . ! or ? (repeated punctuation like '?!' counts once)."""
    raise NotImplementedError


def reading_time(text: str, wpm: int = 200) -> int:
    """Minutes to read, rounded up; at least 1 for non-empty text, 0 for empty."""
    raise NotImplementedError


def slugify(title: str) -> str:
    """Lowercase, accents removed, non-alphanumerics -> single '-', no leading/trailing '-'."""
    raise NotImplementedError
''')

(WS / "test_textstats.py").write_text('''from textstats import *


def test_word_count():
    assert word_count("hello world") == 2
    assert word_count("  spaced   out  ") == 2
    assert word_count("") == 0


def test_top_words():
    assert top_words("b a b c a b", 2) == [("b", 3), ("a", 2)]
    assert top_words("The the THE cat", 1) == [("the", 3)]
    assert top_words("x y", 5) == [("x", 1), ("y", 1)]
    assert top_words("don't stop, don't!", 1) == [("don't", 2)]


def test_sentence_count():
    assert sentence_count("Hi. How are you? Great!") == 3
    assert sentence_count("Really?! Yes.") == 2
    assert sentence_count("no punctuation") == 1
    assert sentence_count("") == 0


def test_reading_time():
    assert reading_time("") == 0
    assert reading_time("word") == 1
    assert reading_time(" ".join(["w"] * 401)) == 3
    assert reading_time(" ".join(["w"] * 100), wpm=50) == 2


def test_slugify():
    assert slugify("Hello, World!") == "hello-world"
    assert slugify("  Café   con Leche ") == "cafe-con-leche"
    assert slugify("--Already--slugged--") == "already-slugged"
    assert slugify("Año 2026: ¿qué viene?") == "ano-2026-que-viene"
''')

FEATURES = [{"id": f, "test": f"test_{f}", "passes": False} for f in ("word_count", "top_words", "sentence_count", "reading_time", "slugify")]
(WS / "features.json").write_text(json.dumps(FEATURES, indent=1))
(WS / "PROGRESS.md").write_text("# Progress log\n(empty: first session)\n")

RUNNER = '''
import importlib, sys, traceback
sys.path.insert(0, ".")
for m in list(sys.modules):
    if m in ("textstats", "test_textstats"): del sys.modules[m]
t = importlib.import_module("test_textstats")
for name in sorted(n for n in dir(t) if n.startswith("test_")):
    try:
        getattr(t, name)(); print("PASS", name)
    except Exception as e:
        print("FAIL", name, "-", type(e).__name__, str(e)[:200] or traceback.format_exc().splitlines()[-1])
'''


def test_report():
    r = subprocess.run([sys.executable, "-c", RUNNER], cwd=WS, capture_output=True, text=True, timeout=30)
    return (r.stdout + r.stderr).strip()


print(test_report())

## 2. Tools, with the harness enforcing the rules
Workspace-only paths (09.02's `_safe`), a precise `edit_file` (exact string replacement, like coding agents use: cheaper and safer than
rewriting whole files), a test runner, and `update_feature` that **re-runs the test** before accepting `passes: true`.

In [ ]:
def _safe(path):
    p = (WS / path).resolve()
    if not p.is_relative_to(WS.resolve()):
        raise PermissionError(f"{path!r} is outside the workspace")
    if p.name == "test_textstats.py":
        raise PermissionError("tests are read-only")
    return p


def list_files() -> str:
    """List files in the workspace."""
    return "\n".join(sorted(p.name for p in WS.iterdir() if p.is_file()))


def read_file(path: str) -> str:
    """Read a text file from the workspace."""
    return (WS / path).resolve().read_text()[:6000] if (WS / path).resolve().is_relative_to(WS.resolve()) else "Error: outside workspace"


def write_file(path: str, content: str) -> str:
    """Create or overwrite a file in the workspace."""
    _safe(path).write_text(content)
    return f"wrote {len(content)} chars to {path}"


def edit_file(path: str, old: str, new: str) -> str:
    """Replace the exact text `old` (must appear exactly once) with `new` in a workspace file."""
    p = _safe(path); text = p.read_text()
    if text.count(old) != 1:
        return f"Error: `old` appears {text.count(old)} times; it must appear exactly once. Read the file and copy the text exactly."
    p.write_text(text.replace(old, new))
    return "ok"


def run_tests() -> str:
    """Run the test suite. Returns PASS/FAIL per test."""
    return test_report()


def update_feature(feature_id: str, passes: bool) -> str:
    """Mark a feature in features.json as passing (verified by the harness: its test must actually pass)."""
    feats = json.loads((WS / "features.json").read_text())
    f = next((f for f in feats if f["id"] == feature_id), None)
    if f is None:
        return f"Error: unknown feature {feature_id}"
    if passes and f"PASS {f['test']}" not in test_report():
        return f"Rejected: {f['test']} does not pass yet."
    f["passes"] = passes
    (WS / "features.json").write_text(json.dumps(feats, indent=1))
    return f"{feature_id} -> passes={passes}"


def append_progress(note: str) -> str:
    """Append a short note to PROGRESS.md for whoever continues this work later (you, in a new session)."""
    with open(WS / "PROGRESS.md", "a") as f:
        f.write(f"- {time.strftime('%H:%M')} {note}\n")
    return "noted"

## 3. Skills: instructions on demand
A skill is a folder with a `SKILL.md`. The system prompt only lists each skill's **name and one-line description**; the agent calls
`load_skill` when one is relevant. Many skills cost almost no context until used (progressive disclosure).

In [ ]:
SKILLS = WS / ".skills"
for name, desc, body in [
    ("python-tdd", "How to work test-first on a Python file: read failures, smallest fix, re-run.",
     "1. Run tests. 2. Pick ONE failing test. 3. Read it carefully: the asserts are the spec, including edge cases.\n"
     "4. Implement with the standard library only (re, collections.Counter, math.ceil, unicodedata). 5. Re-run tests. 6. Mark the feature, log progress."),
    ("unicode-text", "Gotchas for accents, case and slugs in Python.",
     "Strip accents with unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode(). Lowercase after. "
     "Use re.sub(r'[^a-z0-9]+', '-', s).strip('-') for slugs. 'ñ' becomes 'n'."),
]:
    (SKILLS / name).mkdir(parents=True, exist_ok=True)
    (SKILLS / name / "SKILL.md").write_text(f"---\nname: {name}\ndescription: {desc}\n---\n{body}\n")


def skill_index():
    out = []
    for p in sorted(SKILLS.glob("*/SKILL.md")):
        meta = dict(line.split(": ", 1) for line in p.read_text().split("---")[1].strip().splitlines())
        out.append(f"- {meta['name']}: {meta['description']}")
    return "\n".join(out)


loaded_skills = []


def load_skill(name: str) -> str:
    """Load the full instructions of a skill listed in the system prompt."""
    p = SKILLS / name / "SKILL.md"
    loaded_skills.append(name)
    return p.read_text() if p.exists() else f"Error: no skill {name}"


print(skill_index())

## 4. The harness: loop + compaction + budgets
Compaction: when the conversation exceeds a token budget, replace the old middle with an LLM-written summary, keeping the system
prompt, the task and the most recent turns. Everything durable lives in files anyway (code, `features.json`, `PROGRESS.md`).

In [ ]:
def tool_schema(fn):
    props = {n: {"type": {str: "string", int: "integer", bool: "boolean"}.get(p.annotation, "string")} for n, p in inspect.signature(fn).parameters.items()}
    return {"type": "function", "function": {"name": fn.__name__, "description": inspect.getdoc(fn),
                                             "parameters": {"type": "object", "properties": props, "required": list(props)}}}


def approx_tokens(messages):
    return sum(len(json.dumps(m, default=str)) for m in messages) // 4


def llm(messages, tools=None):
    for attempt in range(6):
        try:
            return c.chat.completions.create(model=MODEL, messages=messages, temperature=0, max_tokens=4000,
                                             **({"tools": tools} if tools else {}), **EXTRA)
        except Exception as e:  # free tiers: rate limits, transient 5xx
            wait = 10 * (attempt + 1)
            print(f"   (LLM error {type(e).__name__}, retrying in {wait}s)"); time.sleep(wait)
    raise RuntimeError("LLM unavailable")


def compact(messages, keep_last=6):
    head, middle, tail = messages[:2], messages[2:-keep_last], messages[-keep_last:]
    while tail and tail[0].get("role") == "tool":  # don't orphan tool results from their call
        middle.append(tail.pop(0))
    summary = llm([{"role": "user", "content": "Summarise this agent work log for the agent itself to continue: decisions, what was done, "
                    "what failed and why, what's next. Max 150 words.\n\n" + json.dumps(middle, default=str)[:30000]}]).choices[0].message.content
    return head + [{"role": "user", "content": f"[Summary of earlier work, compacted]\n{summary}"}] + tail


class Agent:
    def __init__(self, tools, system, max_steps=20, compact_at=6000, name="agent"):
        self.tools = {f.__name__: f for f in tools}
        self.schemas = [tool_schema(f) for f in tools]
        self.system, self.max_steps, self.compact_at, self.name = system, max_steps, compact_at, name
        self.stats = {"steps": 0, "tool_calls": 0, "compactions": 0}

    def call(self, tc):
        fn = self.tools.get(tc.function.name)
        try:
            return str(fn(**json.loads(tc.function.arguments or "{}"))) if fn else f"Error: unknown tool {tc.function.name}"
        except Exception as e:
            return f"Error: {type(e).__name__}: {e}"

    def run(self, task):
        messages = [{"role": "system", "content": self.system}, {"role": "user", "content": task}]
        for step in range(self.max_steps):
            if approx_tokens(messages) > self.compact_at:
                messages = compact(messages); self.stats["compactions"] += 1
                print(f"   [{self.name}] compacted context -> {approx_tokens(messages)} tokens")
            msg = llm(messages, self.schemas).choices[0].message
            self.stats["steps"] += 1
            if not msg.tool_calls:
                return msg.content
            messages.append({"role": "assistant", "content": msg.content or "",
                             "tool_calls": [t.model_dump() for t in msg.tool_calls]})
            for tc in msg.tool_calls:
                self.stats["tool_calls"] += 1
                result = self.call(tc)
                print(f"   [{self.name}] {tc.function.name}({tc.function.arguments[:70]}) -> {result[:80]!r}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result[:4000]})
            time.sleep(1)
        if "append_progress" in self.tools:  # out of budget: the HARNESS makes sure a handoff note exists (don't rely on the model)
            log = json.dumps(messages[2:], default=str)[-20000:]  # plain-text transcript: no dangling tool calls to confuse the API
            note = llm([{"role": "user", "content": "This is an agent's work log. In at most 2 lines: what it finished and what to do next.\n\n" + log}])
            append_progress(f"[handoff from {self.name}] " + (note.choices[0].message.content or "").strip().replace("\n", " "))
        return "(step budget exhausted)"

## 5. Session 1 (deliberately short) and session 2 (resumes from files)
Each session starts with a **fresh context**. The only memory between them is the workspace. The system prompt tells the agent how to
start a session ("get your bearings": read progress and features, run tests) and how to end one (log progress) — exactly what a human
engineer does when picking up someone else's branch.

In [ ]:
SYSTEM = f"""You are a careful software engineer working in a small Python workspace, possibly continuing someone else's work.
Start every session by reading PROGRESS.md and features.json and running the tests. Then work on ONE failing feature at a time:
implement it in textstats.py (prefer edit_file), run the tests, and mark it with update_feature only when its test passes.
After each feature, append_progress with one line. Never modify the tests. Stop when all features pass, with a one-line summary.

Skills available (call load_skill(name) to read one when it's relevant):
{skill_index()}"""

TOOLS = [list_files, read_file, write_file, edit_file, run_tests, update_feature, append_progress, load_skill]
TASK = "Make all features in features.json pass."


def status():
    feats = json.loads((WS / "features.json").read_text())
    return f"{sum(f['passes'] for f in feats)}/{len(feats)} features passing"


s1 = Agent(TOOLS, SYSTEM, max_steps=12, name="session-1")
print("SESSION 1 ->", s1.run(TASK)[:200])
print(status(), "|", s1.stats)
print((WS / "PROGRESS.md").read_text())

In [ ]:
s2 = Agent(TOOLS, SYSTEM, max_steps=40, compact_at=2500, name="session-2")  # low on purpose, to see compaction
print("SESSION 2 ->", s2.run(TASK)[:300])
print(status(), "|", s2.stats, "| skills loaded:", loaded_skills)
print(test_report())
print((WS / "PROGRESS.md").read_text())

Session 2 didn't see session 1's conversation, yet it continued: the progress log, the feature list and the code carried the state.
Notice the **handoff note**: when session 1 ran out of steps, the harness (not the model) asked for a two-line summary and wrote it to
`PROGRESS.md`. Models often ignore "log your progress" instructions under pressure; anything that must happen belongs in harness code. The low `compact_at` makes compaction kick in during session 2; the agent keeps going after its history is replaced by a summary.
Look at `skills loaded`: the same skills get loaded again after compactions, because the summary dropped their text. Production harnesses
**re-inject** what must survive compaction (loaded skills, CLAUDE.md/AGENTS.md, the todo list) right after compacting. Production harnesses add: an init script that rebuilds the
environment, `git commit` after each feature (so a bad session can be rolled back), and end-to-end tests (browser automation, 09.08)
because unit tests alone let agents mark features done that don't really work.

## 6. Sub-agents: a fresh context for a side quest
A sub-agent gets its own context and a restricted tool set, does one job and returns a **short report** — the main agent's context only
grows by that report. Here: a read-only reviewer.

In [ ]:
reviewer = Agent([list_files, read_file, run_tests], "You are a strict code reviewer. Read-only tools. Find bugs, unhandled edge cases "
                 "and style problems in textstats.py beyond what the tests check. Reply with at most 5 bullets.", max_steps=6, name="reviewer")


def delegate_review() -> str:
    """Ask a reviewer sub-agent to review textstats.py; returns its short report."""
    return reviewer.run("Review textstats.py.")


report = delegate_review()
print(report)
print("reviewer stats:", reviewer.stats)

In a full harness `delegate_review` is just another tool of the main agent (Claude Code's Task tool, Codex sub-agents), and sub-agents
run in parallel for research ("explore these 5 directories") — at the cost of more total tokens.

## 7. How production coding agents are built
| | Harness pieces |
|---|---|
| **Claude Code** (Anthropic) | terminal agent; tools: read/edit/write/grep/glob/bash/web; permission modes; CLAUDE.md memory; todo list; sub-agents; skills; hooks; auto-compaction; MCP |
| **Codex CLI / cloud** (OpenAI) | sandboxed shell (seatbelt/landlock), `apply_patch` edits, AGENTS.md, approvals, cloud tasks in containers |
| **Cursor / Windsurf / Copilot agent** | IDE-integrated; codebase index (embeddings), background agents |
| **Aider** | git-native: every change is a commit; repo map via tree-sitter |
| **OpenHands / SWE-agent** | open-source research harnesses; Docker sandboxes; agent-computer interface design |

Common lessons from all of them: give the model **few, well-designed tools** (SWE-agent's "ACI"), show **precise feedback** (test output,
linter errors), keep **state in files and git**, and **measure** on real tasks (SWE-bench Verified, Terminal-Bench, your own repo's issues).
Use the **Claude Agent SDK** or **OpenAI Agents SDK** (09.05) to build on a production harness instead of from scratch.

## Exercises
1. Build an agent that works for 30+ minutes on a task with compaction and a progress file: e.g. "implement a CSV → SQLite importer with 15
   tests", run sessions with `max_steps=15` until all pass. Log tokens per session.
2. Add `git` to the harness: initialise a repo in the workspace and commit after every passing feature; add a `revert_last_commit` tool.
3. Give the main agent `delegate_review` as a tool and require a review before the final answer. Does code quality (by your judgement) improve?
4. Measure the effect of skills: run session 2 with and without the skills listed. Compare steps, tool calls and errors.

In [ ]:
# your code here